In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas11-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)
print("Total baris:", df.count())

26/11/14 14:25:54 WARN Utils: Your hostname, HP resolves to a loopback address: 127.0.1.1; using 192.168.0.104 instead (on interface wlo1)
26/11/14 14:25:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/14 14:25:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

Total baris: 2500


In [2]:
# A. Feature Engineering
from pyspark.ml.feature import StringIndexer, VectorAssembler

idx_menu_model = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx").fit(df)
idx_cabang_model = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df)
idx_waktu_model = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx").fit(df)

df_indexed = idx_waktu_model.transform(idx_cabang_model.transform(idx_menu_model.transform(df)))
df_indexed.select("nama_menu", "nama_menu_idx", "cabang", "cabang_idx", "waktu_pesan", "waktu_pesan_idx").show(8)

assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)
df_final = assembler.transform(df_indexed)
df_final.select("features", "total_penjualan").show(5, truncate=False)

+--------------+-------------+----------+----------+-----------+---------------+
|     nama_menu|nama_menu_idx|    cabang|cabang_idx|waktu_pesan|waktu_pesan_idx|
+--------------+-------------+----------+----------+-----------+---------------+
|Kentang Goreng|          5.0|  Semarang|       2.0|      Malam|            1.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Sore|            3.0|
|Kentang Goreng|          5.0|  Semarang|       2.0|      Siang|            2.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Pagi|            0.0|
|    Roti Bakar|          4.0|  Magelang|       0.0|      Malam|            1.0|
|     Americano|          0.0|  Magelang|       0.0|       Pagi|            0.0|
|    Cappuccino|          2.0|  Magelang|       0.0|      Malam|            1.0|
|    Cappuccino|          2.0|  Semarang|       2.0|       Pagi|            0.0|
+--------------+-------------+----------+----------+-----------+---------------+
only showing top 8 rows

+--

In [3]:
# B. Train/Test Split & Pelatihan Model
from pyspark.ml.regression import LinearRegression

train, test = df_final.randomSplit([0.75, 0.25], seed=101)
print(f"Jumlah data training : {train.count()}")  
print(f"Jumlah data testing  : {test.count()}")   

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)

print("Model berhasil dilatih!")
print("Koefisien:", model.coefficients)
print("Intercept:", model.intercept)

Jumlah data training : 1839
Jumlah data testing  : 661
Model berhasil dilatih!
Koefisien: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


In [4]:
# C. Evaluasi Model
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)
predictions.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan",
                   "total_penjualan", "prediction").show(10)

rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse").evaluate(predictions)
r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2").evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

+---------+--------+-----------+---+------------+------------------+------------------+
|nama_menu|  cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|        prediction|
+---------+--------+-----------+---+------------+------------------+------------------+
|Americano|Magelang|      Malam|  1|       25000|19721.170446051627|31056.146105506836|
|Americano|Magelang|      Malam|  1|       25000|  21236.0956379149|31056.146105506836|
|Americano|Magelang|      Malam|  4|       22000|  85834.8564401191| 88127.37331410211|
|Americano|Magelang|      Malam|  4|       22000|102174.09100317833| 88127.37331410211|
|Americano|Magelang|      Malam|  4|       30000|113044.63431555482|112310.74025938749|
|Americano|Magelang|      Malam|  5|       22000|115810.29600214415|110174.03658512786|
|Americano|Magelang|      Malam|  5|       25000|140386.40273615732|119242.79918960987|
|Americano|Magelang|       Pagi|  2|       30000| 73842.64420790318| 67877.77492201277|
|Americano|Magelang|       Pagi|

In [6]:
# D. Eksperimen Perbandingan Fitur
assembler_2 = VectorAssembler(inputCols=["qty", "harga_satuan"], outputCol="features_2")
df_2 = assembler_2.transform(df_final)
train_2, test_2 = df_2.randomSplit([0.75, 0.25], seed=101)

model_2 = LinearRegression(featuresCol="features_2", labelCol="total_penjualan").fit(train_2)
pred_2 = model_2.transform(test_2)

rmse_2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse").evaluate(pred_2)
r2_2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2").evaluate(pred_2)

print(f"{'Model':<32}{'RMSE':>14}{'R²':>10}")
print(f"{'Utama (5 fitur)':<32}{rmse:>14,.0f}{r2:>10.4f}")
print(f"{'Kedua (qty & harga_satuan)':<32}{rmse_2:>14,.0f}{r2_2:>10.4f}")
print(f"Selisih R² (utama - kedua): {r2 - r2_2:+.4f}")

Model                                     RMSE        R²
Utama (5 fitur)                         10,901    0.9107
Kedua (qty & harga_satuan)              10,896    0.9108
Selisih R² (utama - kedua): -0.0001


Berdasarkan hasil perbandingan, model utama yang menggunakan lima fitur memperoleh nilai R² sebesar 0,9107 dan RMSE sebesar 10.901, sedangkan model kedua yang hanya menggunakan fitur qty dan harga_satuan memperoleh R² sebesar 0,9108 dan RMSE sebesar 10.896. Hasil tersebut menunjukkan bahwa model kedua memiliki performa sedikit lebih baik dibandingkan model utama. Selisih nilai R² hanya sebesar -0,0001, sehingga penambahan informasi menu, cabang, dan waktu pemesanan tidak memberikan peningkatan akurasi yang berarti. Dengan demikian, penggunaan dua fitur saja sudah cukup untuk menghasilkan prediksi total penjualan yang hampir sama baiknya.

In [10]:
# E. Prediksi Skenario Baru
skenario = spark.createDataFrame([
    ("Kentang Goreng",  "Semarang",   "Malam",  3, 90000),
    ("Roti Bakar", "Magelang", "Malam", 2, 60000),
    ("Cappuccino", "Semarang",   "Malam", 5, 100000),
], ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"])

# Pakai model StringIndexer & assembler yang SAMA dari Bagian A (bukan yang baru)
skenario_idx = idx_waktu_model.transform(idx_cabang_model.transform(idx_menu_model.transform(skenario)))
skenario_feat = assembler.transform(skenario_idx)

hasil = model.transform(skenario_feat)
hasil.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan",
             (col("qty") * col("harga_satuan")).alias("qty_x_harga"),
             col("prediction").alias("prediksi_total_penjualan")).show(truncate=False)

+--------------+--------+-----------+---+------------+-----------+------------------------+
|nama_menu     |cabang  |waktu_pesan|qty|harga_satuan|qty_x_harga|prediksi_total_penjualan|
+--------------+--------+-----------+---+------------+-----------+------------------------+
|Kentang Goreng|Semarang|Malam      |3  |90000       |270000     |271581.8268473027       |
|Roti Bakar    |Magelang|Malam      |2  |60000       |120000     |158412.18273820978      |
|Cappuccino    |Semarang|Malam      |5  |100000      |500000     |346274.00483892066      |
+--------------+--------+-----------+---+------------+-----------+------------------------+



In [11]:
spark.stop()

print("SparkSession berhasil ditutup.")

SparkSession berhasil ditutup.
